In [1]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import time
import random
import pygame
from loguru import logger
import json
import os

# --- Constants ---
SCREEN_WIDTH = 1200
SCREEN_HEIGHT = 800
FPS = 60

# Colors
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
RED = (220, 50, 50)
GREEN = (50, 220, 50)
BLUE = (50, 50, 220)
GRAY = (150, 150, 150)


In [2]:
import gymnasium as gym
from gymnasium.spaces import Box, Discrete,Tuple
import numpy as np
import pygame

# Define colors
WHITE = (255, 255, 255)
RED = (255, 0, 0)
GREEN = (0, 255, 0)
BLUE = (0, 0, 255)  # Color for the trajectory

class CustomEnv(gym.Env):
    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 60}

    def __init__(self,render_mode=None):
        super().__init__()
        self.grid_size = 10

        self.env_width = self.grid_size*1.5
        self.env_height = self.grid_size

        self.pause = False
        self.domain_randomization = False
        self.render_mode = render_mode

        # Define two separate thresholds for obstacle handling
        self.distance_threshold_penalty = 5  # Penalty zone threshold (larger value)
        self.distance_threshold_collision = 1.5  # Collision threshold (smaller value)
        self.distance_threshold_arm = 3  # Arm threshold (smaller value)
        self.penalty_factor = 5  # Penalty scaling factor
        self.distance_reward_factor = 2
        self.smooth_action_penalty = 2
        self.steps = 0
        self.margin = 0.3
        self.reward_arm = -100
        self.reward_hand = -100
        self.reward_bound = -200
        self.reward_max_step = 200
        self.reward_step = 10
        self.stride_robot_random = [1,3]
        self.stride_hand_random = [0.6,1]
        self.hand_move_epsilon = 0.1


        self.current_distance = 0  # Current distance to goal, used for reward shaping
        self.max_steps = 50  # Set a maximum number of steps to prevent infinite loops
        # Action space (dx, dy)
        self.action_space = Box(low=-1, high=1, shape=(2,), dtype=np.float32)
        # Observation space (robot_x, robot_y, goal_x, goal_y)
        self.observation_shape = 2+2+2+1+1+1+2+1+1+2 # Robot position, hand position, velocity_hand,radius_hand, and distance to hand

        self.observation_space = Box(low=0, high=np.array([self.env_width,self.env_height, self.env_width, self.env_height,1,1 , (2**0.5)*self.grid_size,0.5*self.grid_size,0.5*self.grid_size,2*self.grid_size,self.grid_size,self.stride_robot_random[1],self.stride_hand_random[1],self.env_width,self.env_height]), 
                                     shape=(self.observation_shape,), dtype=np.float32)

        self.random = True
        # For rendering
        self.window = None
        self.clock = None
        self.cell_size = 50 # Pixels per grid unit
        self.trajectory_points = [] # New: List to store past robot positions
        self.dist_arm = 0


    def dist_point_to_segment_correct(self,P, A, B, eps=1e-12):
        P = np.asarray(P, dtype=float)
        A = np.asarray(A, dtype=float)
        B = np.asarray(B, dtype=float)
        v = B - A
        w = P - A
        vv = np.dot(v, v)
        if vv <= eps:
            # A and B coincide: treat as point A
            C = A.copy()
            d = np.linalg.norm(P - A)
            t = 0.0
            case = 'endpoint_A'
        else:
            t = np.dot(w, v) / vv
            if t < 0.0:
                C = A
                d = np.linalg.norm(P - A)
                case = 'before_A'
            elif t > 1.0:
                C = B
                d = np.linalg.norm(P - B)
                case = 'after_B'
            else:
                C = A + t * v
                d = np.linalg.norm(P - C)
                case = 'on_segment'
        return float(d), C, float(t), case

    def _get_obs(self):
        
        return np.concatenate(([self.robot_position]+ 
                               [self.hand_position]+
                               [self.last_action]+
                               [np.array([self.current_distance])]+
                               [np.array([min(self.robot_position[0],
                                              self.robot_position[1],
                                              self.env_width-self.robot_position[0],
                                              self.env_height-self.robot_position[1])])]+
                                [np.array([self.dist_arm])]+
                                [self.fixed_point]+
                                [np.array([self.stride_robot])]+
                                [np.array([self.stride_hand])]+
                                [np.array([self.env_width,self.env_height])]))

    def _get_info(self):
        return {
            "distance_to_hand": self.current_distance,
            "robot_position": self.robot_position,
            "hand_position": self.hand_position,
            'distance_arm':self.dist_arm,
            "fix_point":self.fixed_point,
        }

    def reset(self, seed=None, options=None):

        super().reset()
        self.distance = []
        self.stride_robot = np.random.uniform(*self.stride_robot_random)  # Randomize stride length
        self.stride_hand = np.random.uniform(*self.stride_hand_random)  # Randomize stride length
        # self.stride_robot = 1  # Randomize stride length
        self.distance_threshold_collision = np.random.uniform(2,3)  # Randomize collision threshold
        self.distance_threshold_penalty = np.random.uniform(3, 4)  # Randomize penalty threshold
        
        
        self.noise_obs_sigma = np.random.uniform(0, 0.1)  # Add some noise to observation to make it more realistic
        self.noise_action_sigma = np.random.uniform(0,0.1)  # Add some noise to action to make it more realistic
        
        
        
        self.robot_position = np.random.uniform(self.margin, [self.env_width-self.margin,self.env_height-self.margin])  # Randomize robot position
        self.hand_position = np.random.uniform(self.margin, [self.env_width-self.margin,self.env_height-self.margin])  # Randomize hand position
        # self.hand_position = np.clip(self.hand_position, self.margin, self.grid_size-self.margin)  # Ensure hand stays within grid bounds
        
        # self.hand_move_mode = 'random' if np.random.rand() < 0.1 else 'towards_robot'  # Randomize hand movement mode
        # self.hand_move_mode = 'towards_robot'
        
        self.current_distance = np.linalg.norm(self.robot_position - self.hand_position)
        self.pre_distance = self.current_distance
        self.last_action = np.zeros(2)
        self.steps = 0
        self.trajectory_points = [self.robot_position.copy()] # New: Reset trajectory and add initial position
        
        self.fixed_point = np.array([self.grid_size*random.uniform(0.2,1.3),self.grid_size])
        return self._get_obs(), self._get_info()

    def _reward(self,action):
        terminated = False
        truncated = False
        reward = 0  # Initialize reward
        done_reason = None  # Initialize done reason

        # action regulation penalty
        # reward -= 0.5 * np.sum(np.square(action))  # Penalty for large actions

        self.dist_arm = self.dist_point_to_segment_correct(self.robot_position,self.hand_position, self.fixed_point)[0]
        if self.dist_arm < self.distance_threshold_arm:
            reward += self.reward_arm 
            terminated = True  # Truncate if arm is too short

        # boundary penalty
        if np.any(self.robot_position <= self.margin) or (self.env_height-self.robot_position[1] <=self.margin) or self.env_width-self.robot_position[0] <=self.margin:
            reward += self.reward_bound
            terminated = True  # Truncate if robot goes out of bounds
            done_reason = "out of bounds"

    
        # Auxiliary Rewards -  distance to hand
        self.current_distance = np.linalg.norm(self.robot_position - self.hand_position)
        self.distance.append(self.current_distance)
        reward += (self.current_distance-self.pre_distance)*self.distance_reward_factor  # Reward shaping based on distance change
        self.pre_distance = self.current_distance

        # Obstacle handling with two thresholds
        if self.current_distance < self.distance_threshold_collision:
            reward += self.reward_hand
            terminated = True  # Terminate if too close to obstacles
            done_reason = "collision with obstacle"
        elif self.current_distance < self.distance_threshold_penalty:
            reward -= self.penalty_factor * (self.distance_threshold_penalty - self.current_distance)  # Penalty for being too close to obstacles

        reward -= self.smooth_action_penalty * np.linalg.norm(action - self.last_action)

        # Small reward for each step taken to encourage exploration
        reward+= self.reward_step 

        # Truncate if max steps reached and give max step reward
        if self.steps >= self.max_steps:
            reward += self.reward_max_step
            truncated = True  

        return reward,terminated,truncated,done_reason

    def _get_hand_movement(self):

        # if self.hand_move_mode == 'random':
        #     move_hand = np.random.uniform(-1, 1, size=2)  # Randomly move the hand position slightly
        # elif self.hand_move_mode == 'towards_robot':
        #     dir_vector = self.robot_position - self.hand_position
        #     if np.linalg.norm(dir_vector) > 0:
        #         dir_vector /= np.linalg.norm(dir_vector)
        #     move_hand = dir_vector * self.stride_hand  # Move hand towards robot position
        if random.random() < self.hand_move_epsilon:
            move_hand = np.random.uniform(-1, 1, size=2)  # Randomly move the hand position slightly
        else:
            dir_vector = self.robot_position - self.hand_position
            if np.linalg.norm(dir_vector) > 0:
                dir_vector /= np.linalg.norm(dir_vector)
            move_hand = dir_vector * self.stride_hand  # Move hand towards robot position
        
        return move_hand






    def step(self, action):
        if self.random:
            action+=np.random.normal(0,self.noise_action_sigma,size=self.action_space.shape)  # Add some noise to action to make it more realistic

        move_hand = self._get_hand_movement()
        self.hand_position += move_hand  # Update hand position
        self.hand_position = np.clip(self.hand_position, self.margin, [self.env_width-self.margin,self.env_height-self.margin])  # Ensure hand stays within grid bounds
        # self.fixed_point+= np.array([np.,0])  # Randomize fixed point position

        self.robot_position += action * self.stride_robot  # Scale the action to control speed
        self.trajectory_points.append(self.robot_position.copy()) # New: Add current position to trajectory
        self.steps += 1
        

        reward,terminated,truncated,done_reason = self._reward(action)
        info = self._get_info()
        info['done_reason'] = done_reason
        info['distance_mean'] = np.mean(self.distance)
        observation = self._get_obs()
        if self.random:
            observation += np.random.normal(0, self.noise_obs_sigma, size=self.observation_shape)  # Add some noise to observation to make it more realistic

        return observation, reward, terminated, truncated, info

    def render(self, mode="human"):
 
        pygame.display.init()
        self.window = pygame.display.set_mode(
                (int(self.grid_size * self.cell_size), int(self.grid_size * self.cell_size))
            )
        pygame.display.set_caption("CustomEnv")
        if self.clock is None:
            self.clock = pygame.time.Clock()
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                import sys
                sys.exit() # Exit the program

            elif event.type == pygame.MOUSEBUTTONDOWN:
                mouse_x, mouse_y = event.pos
                self.hand_position = np.array([mouse_x/self.cell_size, mouse_y/self.cell_size])

            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_SPACE:  # 空格键切换暂停
                    self.pause = not self.pause


        canvas = pygame.Surface((self.grid_size * self.cell_size, self.grid_size * self.cell_size))
        canvas.fill(WHITE)
        virus_image = pygame.image.load("hand.png").convert_alpha()  # Load an image if needed, but not used here
        robot_image = pygame.transform.scale(virus_image, (int(self.cell_size * 2), int(self.cell_size * 2)))  # Scale the image
        # New: Draw the trajectory
        if len(self.trajectory_points) > 1:
            scaled_points = []
            for point in self.trajectory_points:
                scaled_points.append((int(point[0] * self.cell_size), int(point[1] * self.cell_size)))
            
            # Draw lines between consecutive points
            pygame.draw.lines(canvas, BLUE, False, scaled_points, 2) # Blue line, not closed, 2 pixels wide
            
            # Optionally, draw small circles at each point to emphasize
            for point_coord in scaled_points:
                pygame.draw.circle(canvas, BLUE, point_coord, 3) # Small blue circles

        # Draw robot
        pygame.draw.circle(
            canvas,
            RED,
            (int(self.robot_position[0] * self.cell_size), int(self.robot_position[1] * self.cell_size)),
            int(self.cell_size * 0.2)
        )
        # Draw obstacles

        canvas.blit(robot_image, (int((self.hand_position[0]-1) * self.cell_size), int((self.hand_position[1]-1) * self.cell_size+1)))
        pygame.draw.circle(canvas,
                            GREEN, 
                            (int((self.hand_position[0]) * self.cell_size), 
                            int((self.hand_position[1]) * self.cell_size+1)), 
        int(self.cell_size * 0.2)
        )

        self.window.blit(canvas, canvas.get_rect())
        pygame.event.pump()
        pygame.display.flip()
        self.clock.tick(self.metadata["render_fps"])
        time.sleep(0.5)
    
    def load_args(self, args):
        pass

    def save_args(self,path):
        env_args = {
            "grid_size": self.grid_size,
            "distance_threshold_penalty":self.distance_threshold_penalty,
            "distance_threshold_collision":self.distance_threshold_collision,
            "penalty_factor":self.penalty_factor,
            "distance_reward_factor":self.distance_reward_factor,
            "smooth_action_penalty":self.smooth_action_penalty,
            "max_steps":self.max_steps,
            "margin":self.margin,
            "reward_step":self.reward_step,
            "reward_max_step":self.reward_max_step,
            "reward_bound":self.reward_bound,
            "reward_arm":self.reward_arm,
            "reward_hand":self.reward_hand,
            "stride_robot_range":self.stride_robot_random,
            "stride_hand_range":self.stride_hand_random,
            "move_hand_epsilon":self.hand_move_epsilon,



        }
        with open(os.path.join(path, "env_args.json"), "w") as f:
            json.dump(env_args, f,indent=4)
        

    def close(self):
        pygame.display.quit()
        pygame.quit()





In [3]:
import numpy as np
from collections import deque

class BiomechanicalFilter:
    def __init__(self, mode='healthy'):
        """
        初始化滤波器
        :param mode: 'healthy', 'parkinson', 'stroke', 'ataxia'
        :param dt: 仿真步长 (秒)
        """
        self.mode = mode
        self.t = 0
        
        # --- 帕金森参数 (基于 Rocon et al. 2004) ---
        self.tremor_amp = 0.4     # 震颤幅度 (像素或单位距离)
        self.tremor_freq = 1/8    # 频率 Hz (4-6Hz 是典型值)
        
        # --- 中风参数 (基于 Rohrer et al. 2002) ---
        self.drag_factor = 0.6    # 肌无力: 只能发挥 60% 的速度
        self.submove_timer = 0    # 子运动计时器
        self.is_stuck = False     # 是否处于"停顿"状态
        
        # --- 共济失调参数 (基于 Manto et al. 1994) ---
        self.dysmetria_gain = 1.5 # 过冲系数 (意向性震颤)
        self.momentum = np.zeros(2) # 惯性/动量
        self.friction = 0.1       # 模拟刹不住车的摩擦系数
        
        # --- 通用: 延迟缓冲区 ---
        self.delay_buffer = deque(maxlen=10) # 模拟反应延迟

    def reset(self):
        self.t = 0
        self.momentum = np.zeros(2)
        self.delay_buffer.clear()
        self.is_stuck = False

    def apply(self, ideal_action, current_pos, target_pos=None):
        """
        核心函数
        :param ideal_action: (dx, dy) 原始动作，假设范围 [-1, 1] 或速度向量
        :param current_pos: 当前手的位置 (x, y)
        :param target_pos: 目标位置 (用于共济失调计算距离)，可选
        :return: (dx, dy) 实际执行的动作
        """
        self.t += 1
        
        # 1. 基础处理：将其转换为 numpy 数组
        action = np.array(ideal_action, dtype=float)

        # 2. 根据模式分发处理
        if self.mode == 'healthy':
            final_action = self._apply_healthy(action)
            
        elif self.mode == 'parkinson':
            final_action = self._apply_parkinson(action)
            
        elif self.mode == 'stroke':
            final_action = self._apply_stroke(action)
            
        elif self.mode == 'ataxia':
            final_action = self._apply_ataxia(action, current_pos, target_pos)
            
        else:
            final_action = action

        # 3. 全局物理约束 (Sim2Real 保护)
        # 防止瞬间速度过大导致物理引擎穿模
        final_action = np.clip(final_action, -10, 10) # 假设最大速度限制
        
        return final_action

    # ---------------- 具体病理实现 ----------------

    def _apply_healthy(self, action):
        # 正常人也有微小的 Minimum Jerk 平滑，这里简化为不做改动
        # 或者加极微小的高斯噪声模拟传感器误差
        return action

    def _apply_parkinson(self, action):
        """
        模型依据: Rocon et al. (2004)
        叠加正弦震颤 + 随机高斯噪声
        """
        # 计算震颤向量
        tremor_x = self.tremor_amp * np.sin(2 * np.pi * self.tremor_freq * self.t)
        tremor_y = self.tremor_amp * np.cos(2 * np.pi * self.tremor_freq * self.t)
        
        # 随机相位漂移 (让震颤看起来不那么机械)
        noise = np.random.normal(0, 0.2, 2)
        
        return action + np.array([tremor_x, tremor_y]) + noise

    def _apply_stroke(self, action):
        """
        模型依据: Rohrer et al. (2002) - Submovements
        表现为: 迟缓 (Lag) + 间歇性停顿 (Stuttering) + 速度削弱
        """
        # 1. 模拟神经延迟 (Lag)
        self.delay_buffer.append(action)
        if len(self.delay_buffer) < 3: # 假设延迟 3 帧
            return np.zeros(2)
        delayed_action = self.delay_buffer.popleft()
        
        # 2. 模拟子运动 (间歇性停顿)
        # 每隔一段时间随机"卡住"一下
        if self.submove_timer <= 0:
            if np.random.rand() < 0.1: # 10% 概率进入停顿
                self.is_stuck = True
                self.submove_timer = np.random.randint(2, 5) # 停顿 5-15 帧
            else:
                self.is_stuck = False
                self.submove_timer = np.random.randint(5, 10) # 正常运动 20-60 帧
        
        self.submove_timer -= 1
        
        if self.is_stuck:
            return delayed_action * 0.1 # 几乎不动
        else:
            # 3. 肌无力 (Weakness)
            return delayed_action * self.drag_factor

    def _apply_ataxia(self, action, current_pos, target_pos):
        """
        模型依据: Manto et al. (1994) - Dysmetria
        表现为: 惯性过大 (刹不住车) + 距离相关的抖动
        """
        # 1. 意向性震颤 (距离目标越近，抖动越大)
        dist_noise = 0
        if target_pos is not None:
            dist = np.linalg.norm(current_pos - target_pos)
            # 距离越近(dist小)，噪声因子反而需要处理，
            # 这里简化为：速度增益误差
        
        # 2. 动量模型 (High Momentum / Low Friction)
        # 新的速度不是直接等于 Action，而是很大程度上受上一步速度影响
        # V_new = V_old * (1 - friction) + Action * Gain
        
        self.momentum = self.momentum * (1 - self.friction) + action * self.dysmetria_gain
        
        return self.momentum
    def _apply_attention(self):
        pass


In [24]:
import gymnasium as gym
from gymnasium.spaces import Box, Discrete,Tuple
import numpy as np
import pygame

# Define colors
WHITE = (255, 255, 255)
RED = (255, 0, 0)
GREEN = (0, 255, 0)
BLUE = (0, 0, 255)  # Color for the trajectory

class CustomEnv_hand(gym.Env):
    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 60}
    hand_mode : str   #  'healthy', 'parkinson', 'stroke', 'ataxia'
    def __init__(self,robot_model,render_mode=None,hand_mode = 'healthy'):
        super().__init__()
        if hand_mode not in ['healthy', 'parkinson', 'stroke', 'ataxia']:
            raise ValueError("Invalid hand mode,must in ['healthy', 'parkinson', 'stroke', 'ataxia']")
        self.grid_size = 10
        self.robot_model = robot_model
        self.last_action_robot = np.zeros(2)
        self.env_width = self.grid_size*1.5
        self.env_height = self.grid_size
        

        self.pause = False
        self.domain_randomization = False
        self.render_mode = render_mode

        # Define two separate thresholds for obstacle handling
        self.distance_threshold_penalty = 5  # Penalty zone threshold (larger value)
        self.distance_threshold_collision = 1.5  # Collision threshold (smaller value)
        self.distance_threshold_arm = 1  # Arm threshold (smaller value)
        self.penalty_factor = -5  # Penalty scaling factor
        self.distance_reward_factor = -2
        self.smooth_action_penalty = -5
        self.steps = 0
        self.margin = 0.3
        self.reward_arm =0
        self.reward_hand = 300
        self.reward_bound = -400
        self.reward_max_step = 0
        self.reward_step = -10
        self.stride_robot_random = [1,3]
        self.stride_hand_random = [0.6,1]
        self.hand_move_epsilon = 0.1


        self.current_distance = 0  # Current distance to goal, used for reward shaping
        self.max_steps = 50  # Set a maximum number of steps to prevent infinite loops
        # Action space (dx, dy)
        self.action_space = Box(low=-1, high=1, shape=(2,), dtype=np.float32)
        # Observation space (robot_x, robot_y, goal_x, goal_y)
        self.observation_shape = 2+2+2+1+1+1+2+1+1+2+2 # Robot position, hand position, velocity_hand,radius_hand, and distance to hand

        self.observation_space = Box(low=0, high=np.array([self.stride_hand_random[1],0.5,1,self.env_width,self.env_height, self.env_width, self.env_height,1,1 , (2**0.5)*self.grid_size,0.5*self.grid_size,0.5*self.grid_size,2*self.grid_size,self.grid_size,self.stride_robot_random[1],self.env_width,self.env_height]), 
                                     shape=(self.observation_shape,), dtype=np.float32)

        self.random = True
        # For rendering
        self.window = None
        self.clock = None
        self.cell_size = 50 # Pixels per grid unit
        self.trajectory_points = [] # New: List to store past robot positions
        self.dist_arm = 0
        self.max_length_arm = self.grid_size

        self.execution_filter = BiomechanicalFilter(hand_mode)

    def calculate_fix_point(self):
        dist_arm = np.linalg.norm(self.fixed_point - self.hand_position)
        # print(dist_arm,self.fixed_point,self.hand_position)

        if dist_arm > self.max_length_arm:
            if self.fixed_point[0] < self.hand_position[0]:
                self.fixed_point[0] = self.hand_position[0] - np.sqrt(self.max_length_arm**2-(self.fixed_point[1]-self.hand_position[1])**2)
            else:
                self.fixed_point[0] = self.hand_position[0] + np.sqrt(self.max_length_arm**2-(self.fixed_point[1]-self.hand_position[1])**2)

        # === 核心改进 2: 生物力学成本 (防平扫) ===
    def _calculate_biomechanical_cost(self, current_pos, proposed_movement):
        """
        计算动作的生物力学成本，惩罚远端的切向运动(平扫)
        """
        # 1. 手臂向量 (Base -> Hand)
        arm_vec = current_pos - self.fixed_point
        arm_len = np.linalg.norm(arm_vec)
        
        if arm_len < 1e-4: return 0
        
        # 2. 动作分解
        arm_dir = arm_vec / arm_len # 单位向量
        
        # 径向速度 (伸缩)
        v_radial_scalar = np.dot(proposed_movement, arm_dir)
        v_radial = v_radial_scalar * arm_dir
        
        # 切向速度 (横扫)
        v_tangential = proposed_movement - v_radial
        
        speed_tan = np.linalg.norm(v_tangential)
        speed_rad = abs(v_radial_scalar)
        
        # 3. 惩罚公式
        # 切向惩罚：随臂长平方增长
        penalty_sweep =  20* (speed_tan**2) * (1 + 2.0 * (arm_len / self.max_length_arm)**2)
        
        # 径向惩罚：很小，允许伸缩
        penalty_effort =  5* (speed_rad**2)
        
        return penalty_sweep + penalty_effort
    


    def dist_point_to_segment_correct(self,P, A, B, eps=1e-12):
        P = np.asarray(P, dtype=float)
        A = np.asarray(A, dtype=float)
        B = np.asarray(B, dtype=float)
        v = B - A
        w = P - A
        vv = np.dot(v, v)
        if vv <= eps:
            # A and B coincide: treat as point A
            C = A.copy()
            d = np.linalg.norm(P - A)
            t = 0.0
            case = 'endpoint_A'
        else:
            t = np.dot(w, v) / vv
            if t < 0.0:
                C = A
                d = np.linalg.norm(P - A)
                case = 'before_A'
            elif t > 1.0:
                C = B
                d = np.linalg.norm(P - B)
                case = 'after_B'
            else:
                C = A + t * v
                d = np.linalg.norm(P - C)
                case = 'on_segment'
        return float(d), C, float(t), case

    def _get_obs(self):
        
        return np.concatenate(( [np.array([self.stride_hand])]+
                               [np.array([self.random_epsilon])]+
                               [np.array([self.direction_error])]+
                               
                               
                                [self.robot_position]+ 
                               [self.hand_position]+ 
                               [self.last_action]+
                               [np.array([self.current_distance])]+
                               [np.array([min(self.hand_position[0],
                                              self.hand_position[1],
                                              self.env_width-self.hand_position[0],
                                              self.env_height-self.hand_position[1])])]+
                                [np.array([self.dist_arm])]+
                                [self.fixed_point]+
                                [np.array([self.stride_robot])]+
                                [np.array([self.env_width,self.env_height])]))
    
    def _get_obs_robot(self):
        
        return np.concatenate(([self.robot_position]+ 
                               [self.hand_position]+
                               [self.last_action_robot]+
                               [np.array([self.current_distance])]+
                               [np.array([min(self.robot_position[0],
                                              self.robot_position[1],
                                              self.env_width-self.robot_position[0],
                                              self.env_height-self.robot_position[1])])]+
                                [np.array([self.dist_arm])]+
                                [self.fixed_point]+
                                [np.array([self.stride_robot])]+
                                [np.array([self.stride_hand])]+
                                [np.array([self.env_width,self.env_height])]))

    def _get_info(self):
        return {
            "distance_to_hand": self.current_distance,
            "robot_position": self.robot_position,
            "hand_position": self.hand_position,
            'distance_arm':self.dist_arm,
            "fix_point":self.fixed_point,
        }

    def reset(self, seed=None, options=None):

        super().reset()
        self.distance = []
        self.stride_robot = np.random.uniform(*self.stride_robot_random)  # Randomize stride length
        self.stride_hand = np.random.uniform(*self.stride_hand_random)  # Randomize stride length
        # self.stride_robot = 1  # Randomize stride length
        self.distance_threshold_collision = np.random.uniform(1.5,2.5)  # Randomize collision threshold
        self.distance_threshold_penalty = np.random.uniform(3, 4)  # Randomize penalty threshold
        
        self.random_epsilon = np.random.uniform(0,0.5)  # Randomize epsilon for randomization
        self.direction_error = np.random.uniform(0,1)  # Randomize direction error for randomization
        
        self.noise_obs_sigma = np.random.uniform(0, 0.1)  # Add some noise to observation to make it more realistic
        self.noise_action_sigma = np.random.uniform(0,0.1)  # Add some noise to action to make it more realistic
        
        
        
        self.robot_position = np.random.uniform(self.margin, [self.env_width-self.margin,self.env_height-self.margin])  # Randomize robot position
        self.hand_position = np.random.uniform(self.margin, [self.env_width-self.margin,self.env_height-self.margin])  # Randomize hand position
        # self.hand_position = np.clip(self.hand_position, self.margin, self.grid_size-self.margin)  # Ensure hand stays within grid bounds
        
        # self.hand_move_mode = 'random' if np.random.rand() < 0.1 else 'towards_robot'  # Randomize hand movement mode
        # self.hand_move_mode = 'towards_robot'
        
        self.current_distance = np.linalg.norm(self.robot_position - self.hand_position)
        self.pre_distance = self.current_distance
        self.last_action = np.zeros(2)
        self.steps = 0
        self.trajectory_points = [self.robot_position.copy()] # New: Reset trajectory and add initial position
        
        self.fixed_point = np.array([self.grid_size*random.uniform(0.2,1.3),self.grid_size])
        return self._get_obs(), self._get_info()

    def _reward(self,action):
        terminated = False
        truncated = False
        reward = 0  # Initialize reward
        done_reason = None  # Initialize done reason



        self.dist_arm = self.dist_point_to_segment_correct(self.robot_position,self.hand_position, self.fixed_point)[0]
        if self.dist_arm < self.distance_threshold_arm:
            reward += self.reward_arm 
            terminated = True  # Truncate if arm is too short

        # boundary penalty
        if np.any(self.robot_position <= self.margin) or (self.env_height-self.robot_position[1] <=self.margin) or self.env_width-self.robot_position[0] <=self.margin:
            reward += self.reward_bound
            terminated = True  # Truncate if robot goes out of bounds
            done_reason = "out of bounds"

    
        # Auxiliary Rewards -  distance to hand
        self.current_distance = np.linalg.norm(self.robot_position - self.hand_position)
        self.distance.append(self.current_distance)
        reward += (self.current_distance-self.pre_distance)*self.distance_reward_factor  # Reward shaping based on distance change
        self.pre_distance = self.current_distance

        # Obstacle handling with two thresholds
        if self.current_distance < self.distance_threshold_collision:
            reward += self.reward_hand
            terminated = True  # Terminate if too close to obstacles
            done_reason = "collision with obstacle"
        elif self.current_distance < self.distance_threshold_penalty:
            reward -= self.penalty_factor * (self.distance_threshold_penalty - self.current_distance)  # Penalty for being too close to obstacles

        reward -= self.smooth_action_penalty * np.linalg.norm(action - self.last_action)

        # Small reward for each step taken to encourage exploration
        reward+= self.reward_step 

        # Truncate if max steps reached and give max step reward
        if self.steps >= self.max_steps:
            reward += self.reward_max_step
            truncated = True  

        return reward,terminated,truncated,done_reason

    def _get_hand_movement(self):

        # if self.hand_move_mode == 'random':
        #     move_hand = np.random.uniform(-1, 1, size=2)  # Randomly move the hand position slightly
        # elif self.hand_move_mode == 'towards_robot':
        #     dir_vector = self.robot_position - self.hand_position
        #     if np.linalg.norm(dir_vector) > 0:
        #         dir_vector /= np.linalg.norm(dir_vector)
        #     move_hand = dir_vector * self.stride_hand  # Move hand towards robot position
        if random.random() < self.hand_move_epsilon:
            move_hand = np.random.uniform(-1, 1, size=2)  # Randomly move the hand position slightly
        else:
            dir_vector = self.robot_position - self.hand_position
            if np.linalg.norm(dir_vector) > 0:
                dir_vector /= np.linalg.norm(dir_vector)
            move_hand = dir_vector * self.stride_hand  # Move hand towards robot position
        
        return move_hand
    


    def _check_line_intersection(self, p1, p2, p3, p4):
        """
        判断线段 p1-p2 (机器人路径) 和 p3-p4 (手臂) 是否相交
        """
        def ccw(A, B, C):
            # 判断三个点的方向 (Counter-Clockwise)
            return (C[1] - A[1]) * (B[0] - A[0]) > (B[1] - A[1]) * (C[0] - A[0])

        # 如果 p1-p2 的两个端点在 p3-p4 两侧，且 p3-p4 的两个端点在 p1-p2 两侧，则相交
        return ccw(p1, p3, p4) != ccw(p2, p3, p4) and ccw(p1, p2, p3) != ccw(p1, p2, p4)




    def step(self, action):
        # --- 1. 手掌移动逻辑 (保持不变) ---
        if self.random:
            action += np.random.normal(0, self.noise_action_sigma, size=self.action_space.shape)
        
        if random.random() < self.random_epsilon:
            action = np.random.uniform(-1, 1, size=self.action_space.shape)
            move_hand = action * self.stride_hand
        else:
            # angle = 0.1 # 简化了你的原始代码
            # R = np.array([
            #     [np.cos(angle), -np.sin(angle)],
            #     [np.sin(angle),  np.cos(angle)]
            # ])
            # move_hand = R @ (action * self.stride_hand)
            move_hand = action * self.stride_hand
        
        bio_cost = self._calculate_biomechanical_cost(self.hand_position, move_hand)
        
        move_hand = self.execution_filter.apply(move_hand, self.hand_position)  
        self.hand_position += move_hand
        self.hand_position = np.clip(self.hand_position, self.margin, [self.env_width-self.margin, self.env_height-self.margin])
        
        # --- 2. 机器人移动逻辑 (修改部分) ---
        
        # [关键步骤 A]：在移动前记录机器人旧位置
        old_robot_position = self.robot_position.copy()

        action_robot, _ = self.robot_model.predict(self._get_obs_robot())
        self.last_action_robot = action_robot.copy()
        
        # 更新机器人位置
        self.robot_position += action_robot * self.stride_robot
        self.trajectory_points.append(self.robot_position.copy())
        self.steps += 1
        
        # 更新手臂固定点 (确保这是最新的手臂位置)
        self.calculate_fix_point()
        
        # --- 3. 奖励计算与状态判断 ---
        reward, terminated, truncated, done_reason = self._reward(action)
        reward -= bio_cost

        # [关键步骤 B]：碰撞检测逻辑
        # 线段1: 机器人路径 (old_robot_position -> self.robot_position)
        # 线段2: 手臂障碍 (self.fix_point -> self.hand_position)
        
        if self._check_line_intersection(old_robot_position, self.robot_position, self.fixed_point, self.hand_position):
            terminated = True         # 强制结束回合
            reward -= 0          # [可选] 给予较大的碰撞惩罚
            done_reason = "collision_arm" # 更新结束原因
            # print("Robot hit the arm!")

        self.last_action = action.copy()
        info = self._get_info()
        info['done_reason'] = done_reason
        info['distance_mean'] = np.mean(self.distance)
        observation = self._get_obs()
        
        return observation, reward, terminated, truncated, info

    def render(self, mode="human"):
 
        pygame.display.init()
        self.window = pygame.display.set_mode(
                (int(self.grid_size * self.cell_size), int(self.grid_size * self.cell_size))
            )
        pygame.display.set_caption("CustomEnv")
        if self.clock is None:
            self.clock = pygame.time.Clock()
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                import sys
                sys.exit() # Exit the program

            elif event.type == pygame.MOUSEBUTTONDOWN:
                mouse_x, mouse_y = event.pos
                self.hand_position = np.array([mouse_x/self.cell_size, mouse_y/self.cell_size])

            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_SPACE:  # 空格键切换暂停
                    self.pause = not self.pause


        canvas = pygame.Surface((self.grid_size * self.cell_size, self.grid_size * self.cell_size))
        canvas.fill(WHITE)
        virus_image = pygame.image.load("hand.png").convert_alpha()  # Load an image if needed, but not used here
        robot_image = pygame.transform.scale(virus_image, (int(self.cell_size * 2), int(self.cell_size * 2)))  # Scale the image
        # New: Draw the trajectory
        if len(self.trajectory_points) > 1:
            scaled_points = []
            for point in self.trajectory_points:
                scaled_points.append((int(point[0] * self.cell_size), int(point[1] * self.cell_size)))
            
            # Draw lines between consecutive points
            pygame.draw.lines(canvas, BLUE, False, scaled_points, 2) # Blue line, not closed, 2 pixels wide
            
            # Optionally, draw small circles at each point to emphasize
            for point_coord in scaled_points:
                pygame.draw.circle(canvas, BLUE, point_coord, 3) # Small blue circles

        # Draw robot
        pygame.draw.circle(
            canvas,
            RED,
            (int(self.robot_position[0] * self.cell_size), int(self.robot_position[1] * self.cell_size)),
            int(self.cell_size * 0.2)
        )
        # Draw obstacles

        canvas.blit(robot_image, (int((self.hand_position[0]-1) * self.cell_size), int((self.hand_position[1]-1) * self.cell_size+1)))
        pygame.draw.circle(canvas,
                            GREEN, 
                            (int((self.hand_position[0]) * self.cell_size), 
                            int((self.hand_position[1]) * self.cell_size+1)), 
        int(self.cell_size * 0.2)
        )

        self.window.blit(canvas, canvas.get_rect())
        pygame.event.pump()
        pygame.display.flip()
        self.clock.tick(self.metadata["render_fps"])
        time.sleep(0.5)
    
    def load_args(self, args):
        pass

    def save_args(self,path):
        env_args = {
            "grid_size": self.grid_size,
            "distance_threshold_penalty":self.distance_threshold_penalty,
            "distance_threshold_collision":self.distance_threshold_collision,
            "penalty_factor":self.penalty_factor,
            "distance_reward_factor":self.distance_reward_factor,
            "smooth_action_penalty":self.smooth_action_penalty,
            "max_steps":self.max_steps,
            "margin":self.margin,
            "reward_step":self.reward_step,
            "reward_max_step":self.reward_max_step,
            "reward_bound":self.reward_bound,
            "reward_arm":self.reward_arm,
            "reward_hand":self.reward_hand,
            "stride_robot_range":self.stride_robot_random,
            "stride_hand_range":self.stride_hand_random,
            "move_hand_epsilon":self.hand_move_epsilon,



        }
        with open(os.path.join(path, "env_args.json"), "w") as f:
            json.dump(env_args, f,indent=4)
        

    def close(self):
        pygame.display.quit()
        pygame.quit()





In [5]:
def render_environment(robot_position, hand_position, fix_point,trajectory_points, grid_size=10, cell_size=50):
    WHITE = (255, 255, 255)
    RED = (255, 0, 0)
    GREEN = (0, 255, 0)
    BLUE = (0, 0, 255)
    # print(robot_position, hand_position)
    pygame.init()
    window = pygame.display.set_mode((grid_size * cell_size*1.5, grid_size * cell_size))
    canvas = pygame.Surface((grid_size * cell_size*1.5, grid_size * cell_size))
    canvas.fill(WHITE)
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            pygame.quit()
            import sys
            sys.exit() # Exit the program

        elif event.type == pygame.MOUSEBUTTONDOWN:
            mouse_x, mouse_y = event.pos
            hand_position = np.array([mouse_x/cell_size, mouse_y/cell_size])

    if len(trajectory_points) > 1:
        scaled_points = [(int(point[0] * cell_size), int(point[1] * cell_size)) for point in trajectory_points]
        pygame.draw.lines(canvas, BLUE, False, scaled_points, 2)
        for point_coord in scaled_points:
            pygame.draw.circle(canvas, BLUE, point_coord, 3)

    pygame.draw.lines(canvas, (255, 224, 189), False,[hand_position*cell_size, [fix_point[0]*cell_size,fix_point[1]*cell_size]],width=25)

    virus_image = pygame.image.load("../hand.png").convert_alpha()  # Load an image if needed, but not used here
    robot_image = pygame.transform.scale(virus_image, (int(cell_size * 2), int(cell_size * 2)))  # Scale the 
    pygame.draw.circle(canvas, RED, (int(robot_position[0] * cell_size), int(robot_position[1] * cell_size)), int(cell_size * 0.2))
    pygame.draw.circle(canvas, GREEN, (int(hand_position[0] * cell_size), int(hand_position[1] * cell_size)), int(cell_size * 0.2))
    canvas.blit(robot_image, (int((hand_position[0]-1) * cell_size), int((hand_position[1]-1) * cell_size)))
    font = pygame.font.Font(None, 24)
    text = font.render(f"{hand_position[0]},{hand_position[1]}", True, BLUE)
    text_rect = text.get_rect()
    text_rect.center = (int(hand_position[0] * cell_size), int(hand_position[1] * cell_size))
    window.blit(canvas, canvas.get_rect())
    # window.blit(text, text_rect)
    pygame.display.flip()

    return hand_position

In [6]:
from collections import deque
from stable_baselines3.common.callbacks import BaseCallback, EvalCallback, CallbackList
class DebugCallback(BaseCallback):
    def __init__(self, env, render_freq=10000, n_episodes=1, log_freq=10000, verbose=1):
        super().__init__(verbose)
        self.log_freq = log_freq
        # 用deque统计最近N个done的终止原因，避免内存爆炸
        self.termination_reasons = deque(maxlen=1000)  # 统计最近1000次终止
        self.env_to_render = env
        self.render_freq = render_freq
        self.n_episodes = n_episodes
        self.distance_mean = deque(maxlen=1000) 

    def _on_step(self) -> bool:
        # 先从env info里读取终止原因
        # print((self.locals.keys()))
        infos = self.locals.get('infos', None)

        dones = self.locals.get('dones', None)
        if infos is not None and dones is not None:
            for done, info in zip(dones, infos):
                if done and info is not None and 'done_reason' in info:
                    self.termination_reasons.append(info['done_reason'])
                    self.distance_mean.append(info['distance_mean'])

        # 每log_freq步打印信息
        # if self.num_timesteps % self.render_freq == 0 and self.verbose:
        #     for ep in range(self.n_episodes):
        #         obs = self.env_to_render.reset()
        #         done = False
        #         while not done:
        #             action, _states = self.model.predict(obs, deterministic=True)
        #             obs, rewards, done, info = self.env_to_render.step(action)
        #             self.env_to_render.render()
        #             time.sleep(0.6)

        #             if done:

        #                 self.env_to_render.close()
        #                 break





        if self.num_timesteps % self.log_freq == 0 and self.verbose:
            log = self.model.logger.name_to_value
            ep_rew = log.get('rollout/ep_rew_mean', None)
            ep_len = log.get('rollout/ep_len_mean', None)
            loss = log.get('train/loss', None)
            v_loss = log.get('train/value_loss', None)
            p_loss = log.get('train/policy_gradient_loss', None)
            ent_loss = log.get('train/entropy_loss', None)
            kl = log.get('train/approx_kl', None)

            # 统计终止原因比例
            total = len(self.termination_reasons)
            if total > 0:
                count_hand = sum(1 for r in self.termination_reasons if r == 'out of bounds')
                ratio_hand = count_hand / total
            else:
                ratio_hand = 0.0
            distance_mean = sum(self.distance_mean) / len(self.distance_mean) if len(self.distance_mean) > 0 else 0.0

            # print(f"[{self.num_timesteps:7d}] ep_rew_mean={ep_rew}, ep_len_mean={ep_len}, loss={loss:.3f}, "
            #       f"v_loss={v_loss:.3f}, p_loss={p_loss:.3f}, ent_loss={ent_loss:.3f}, kl={kl:.4f}, "
            #       f"termination_reason_hand_ratio={ratio_hand:.3f}")
            self.logger.record("custom/termination_reason_ratio", ratio_hand)

            self.logger.record("custom/distance_mean", distance_mean)
            self.logger.dump(step=self.num_timesteps)

        return True


In [7]:
# from matplotlib import pyplot as plt

# def metric(trajectory):
#     """
#     trajectory: list of tuples, each tuple contains (observation, action, hand_movement, reward)
#     """
    
#     if not isinstance(trajectory, list):
#         raise TypeError("trajectory should be a list of tuples")

#     # distance = [x[6] for x in trajectory]
#     # return distance
    
#     # plt.hist(distance, bins=10, density=True,edgecolor='black', alpha=0.7,color='skyblue')
#     # plt.xlabel('distance')
#     # plt.ylabel('density')
#     # plt.title('distance distribution')
#     # plt.pause(0.1)

#     # distance_aproximity
#     for item in trajectory:
#         obs = item[0]
#         distance = obs[0:2]




#     # direction_alignment
#     for item in trajectory:
#         obs = item[0]
#         direction = 
#         hand_movement = item[2]

#         position_robot,position_hand = obs[0:2],obs[2:4]
#         direction = position_robot - position_hand
#         consine_angle = np.dot(direction, hand_movement) / (np.linalg.norm(direction) * np.linalg.norm(hand_movement))
#         angle = np.arccos(consine_angle)

#     # reaction_time

    



In [25]:
import gymnasium as gym
import torch
from stable_baselines3 import PPO,SAC
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize, VecMonitor
from stable_baselines3.common.monitor import Monitor
# from stable_baselines3.commom.buffers import ReplayBuffer
from stable_baselines3.common.logger import configure
from stable_baselines3.common.utils import safe_mean

env_raw = CustomEnv()
save_path = "logs/best_model_sac"
i=1
while os.path.exists(save_path+str(i)):
    i+=1
save_path = f"{save_path}{i}"
os.makedirs(save_path)

tensorboard_dir = "./sac_custom_env_tensorboard"
os.makedirs(tensorboard_dir, exist_ok=True)
tensorboard_log_dir = tensorboard_dir + '/'+ f'best_model_sac{i}'
load_model = "logs/best_model_sac88/best_model.zip"





env =DummyVecEnv([lambda: Monitor(env_raw)])  # Wrap the environment with Monitor for logging

# env = VecNormalize(env, norm_obs=True, norm_reward=True)
policy_kwargs = dict(
    net_arch=[dict(pi=[128,256,256,128], qf=[128,256,256,128])],
    # activation_fn=torch.nn.ReLU  # 改为 ReLU，通常更适合稀疏奖励
)
policy_kwargs = dict(
    net_arch=[128,256,256,128],
    # activation_fn=torch.nn.ReLU  # 改为 ReLU，通常更适合稀疏奖励
)



model = SAC("MlpPolicy", env, verbose=1,ent_coef='auto',policy_kwargs=policy_kwargs,tensorboard_log=tensorboard_log_dir)
replay_buffer = model.replay_buffer
model_robot = SAC.load(load_model, env, verbose=1,ent_coef='auto',learning_rate=0.0001,tensorboard_log=tensorboard_log_dir)


env = DummyVecEnv([lambda: Monitor(CustomEnv_hand(robot_model=model_robot))])

model = SAC("MlpPolicy", env, verbose=1,ent_coef='auto',policy_kwargs=policy_kwargs,tensorboard_log=tensorboard_log_dir)

eval_callback = EvalCallback(
    env,
    best_model_save_path=save_path,
    log_path = './logs/',
    eval_freq=10000,  # 每1000步评估一次
    deterministic=True,
    render=True,
    n_eval_episodes=10,  # 每次评估5个episode
)



debug_callback = DebugCallback(env=env,log_freq=10000, verbose=1)
callback = CallbackList([eval_callback, debug_callback])

model.learn(total_timesteps=300000, callback=callback)

settings = {
    'load_model':load_model,
    'tensorboard_log' :tensorboard_log_dir,
           }
env_raw.save_args(save_path)
with open(os.path.join(save_path, "settings.json"), "w") as f:
    json.dump(settings, f)

env.close()


Using cpu device
Using cpu device
Logging to ./sac_custom_env_tensorboard/best_model_sac180\SAC_2


g:\anaconda\envs\RL\lib\site-packages\gymnasium\spaces\box.py:305: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 37.8     |
|    ep_rew_mean     | -475     |
| time/              |          |
|    episodes        | 4        |
|    fps             | 170      |
|    time_elapsed    | 0        |
|    total_timesteps | 151      |
| train/             |          |
|    actor_loss      | 9.43     |
|    critic_loss     | 66       |
|    ent_coef        | 0.985    |
|    ent_coef_loss   | -0.0496  |
|    learning_rate   | 0.0003   |
|    n_updates       | 50       |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 39.2     |
|    ep_rew_mean     | -512     |
| time/              |          |
|    episodes        | 8        |
|    fps             | 92       |
|    time_elapsed    | 3        |
|    total_timesteps | 314      |
| train/             |          |
|    actor_loss      | 17.1     |
|    critic_loss     | 76.3     |
|    ent_coef 

g:\anaconda\envs\RL\lib\site-packages\stable_baselines3\common\vec_env\base_vec_env.py:259: UserWarning: You tried to call render() but no `render_mode` was passed to the env constructor.
  warnings.warn("You tried to call render() but no `render_mode` was passed to the env constructor.")


Eval num_timesteps=10000, episode_reward=-342.54 +/- 168.12
Episode length: 32.40 +/- 17.98
---------------------------------
| eval/              |          |
|    mean_ep_length  | 32.4     |
|    mean_reward     | -343     |
| time/              |          |
|    total_timesteps | 10000    |
| train/             |          |
|    actor_loss      | 389      |
|    critic_loss     | 981      |
|    ent_coef        | 0.172    |
|    ent_coef_loss   | 0.137    |
|    learning_rate   | 0.0003   |
|    n_updates       | 9899     |
---------------------------------
New best mean reward!
------------------------------------------
| custom/                     |          |
|    distance_mean            | 6.77     |
|    termination_reason_ratio | 0.016    |
------------------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 35.8     |
|    ep_rew_mean     | -389     |
| time/              |          |
|    episodes        | 25

KeyboardInterrupt: 

In [84]:
from stable_baselines3 import SAC,PPO
# env =DummyVecEnv([lambda: Monitor(CustomEnv())])  # Wrap the environment with Monitor for logging

# env = VecNormalize.load('vec_norm.pkl',env)
# env = VecNormalize(env)  # Apply normalization to the environment
# seed = np.random.randint(0,1000)
# from custom_env import CustomEnv
# env = CustomEnv_hand(robot_model=model_robot)
pygame.init()
grid_s = 10
cell_s = 50
screen = pygame.display.set_mode((int(grid_s * cell_s * 1.5), int(grid_s * cell_s)))
env.random = False

env = CustomEnv()
model_robot= SAC.load("logs/best_model_sac88/best_model.zip",env=env)  # Load the best model

# ['healthy', 'parkinson', 'stroke', 'ataxia']
env = CustomEnv_hand(robot_model=model_robot,hand_mode='stroke')
model = SAC.load("logs/best_model_sac173/best_model.zip",env=env)  # Load the best model

obs,_ = env.reset()
env.random_epsilon = 0.1

env.stride_hand = 1
env.stride_robot = 2.5
env.distance_threshold_collision = 1.5
state_history = []
max_steps = 40000

for i in range(max_steps):
    obs = env._get_obs()
    info = env._get_info()
    print(env.fixed_point)
    action, _states = model.predict(obs, deterministic=True)
    # print(11111111,action)
    obs, reward, teminated,_, info = env.step(action)
    state_history.append(obs)
    # print(f"obs:{obs}")
    # print(f"action:{action}")



    # print(info["robot_position"],info["hand_position"])
    # print("Reward:", reward)
    # print("distance_arm:",info['distance_arm'])
    # env.render()
    render_aesthetic(info["robot_position"], info["hand_position"],info["fix_point"], trajectory_points=env.trajectory_points,window=screen)
    time.sleep(0.5)  # Control the frame rate
    if teminated:
        render_aesthetic(info["robot_position"], info["hand_position"],info["fix_point"], trajectory_points=env.trajectory_points)
        # time.sleep(0.6)
        env.close()
        break
        print("Resetting environment")
# metric(state_history)

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
[ 2.00946432 10.        ]
[ 2.00946432 10.        ]
[ 2.00946432 10.        ]
[ 2.00946432 10.        ]
[ 2.00946432 10.        ]
[ 2.00946432 10.        ]
[ 2.00946432 10.        ]
[ 2.00946432 10.        ]
[ 2.00946432 10.        ]
[ 2.00946432 10.        ]
[ 2.00946432 10.        ]
[ 2.00946432 10.        ]
[ 2.00946432 10.        ]
[ 2.00946432 10.        ]
[ 2.00946432 10.        ]
[ 2.00946432 10.        ]
[ 2.45865461 10.        ]
[ 2.46166396 10.        ]
[ 2.60533527 10.        ]
[ 2.62831758 10.        ]
[ 3.17085799 10.        ]
[ 3.29837389 10.        ]
[ 3.29837389 10.        ]
[ 3.29837389 10.        ]
[ 3.50091596 10.        ]
[ 4.41704476 10.        ]
[ 4.41704476 10.        ]


SystemExit: 

dict_keys(['grid_size', 'env_width', 'env_height', 'pause', 'domain_randomization', 'render_mode', 'distance_threshold_penalty', 'distance_threshold_collision', 'distance_threshold_arm', 'penalty_factor', 'distance_reward_factor', 'smooth_action_penalty', 'steps', 'margin', 'reward_arm', 'reward_hand', 'reward_bound', 'reward_max_step', 'reward_step', 'stride_robot_random', 'stride_hand_random', 'hand_move_epsilon', 'current_distance', 'max_steps', 'action_space', 'observation_shape', 'observation_space', 'random', 'window', 'clock', 'cell_size', 'trajectory_points', 'dist_arm'])
dict_keys(['grid_size', 'env_width', 'env_height', 'pause', 'domain_randomization', 'render_mode', 'distance_threshold_penalty', 'distance_threshold_collision', 'distance_threshold_arm', 'penalty_factor', 'distance_reward_factor', 'smooth_action_penalty', 'steps', 'margin', 'reward_arm', 'reward_hand', 'reward_bound', 'reward_max_step', 'reward_step', 'stride_robot_random', 'stride_hand_random', 'hand_move_eps

In [11]:
import math
import pygame
import pygame.gfxdraw
import numpy as np

def draw_capsule_rotated(surf, color, center_x, center_y, width, length, angle_deg):
    """
    画一个胶囊：
    angle_deg 是胶囊延伸的方向。
    """
    rad = math.radians(angle_deg)
    cos_a = math.cos(rad)
    sin_a = math.sin(rad)

    # 胶囊底部中心 (center_x, y)
    # 胶囊顶部中心 (沿着 angle 方向延伸 length 长度)
    x2 = center_x + length * cos_a
    y2 = center_y + length * sin_a

    # 计算垂直于延伸方向的宽度向量
    dx = (width / 2) * math.sin(rad)
    dy = (width / 2) * math.cos(rad)

    # 4个角点
    points = [
        (center_x - dx, center_y + dy), # 左底
        (center_x + dx, center_y - dy), # 右底
        (x2 + dx, y2 - dy),             # 右顶
        (x2 - dx, y2 + dy)              # 左顶
    ]
    
    pygame.gfxdraw.aapolygon(surf, points, color)
    pygame.gfxdraw.filled_polygon(surf, points, color)
    
    # 两个圆头
    pygame.gfxdraw.aacircle(surf, int(center_x), int(center_y), int(width/2), color)
    pygame.gfxdraw.filled_circle(surf, int(center_x), int(center_y), int(width/2), color)
    pygame.gfxdraw.aacircle(surf, int(x2), int(y2), int(width/2), color)
    pygame.gfxdraw.filled_circle(surf, int(x2), int(y2), int(width/2), color)

def draw_detailed_hand(surf, fill_color, border_color, center, radius, angle_deg=0):
    """
    绘制一个结构清晰的手掌（带指甲，明确表示手背向上）
    """
    cx, cy = center
    base_rad = math.radians(angle_deg)
    
    # 指甲颜色：比肤色更淡、更白一点
    # 假设 fill_color 是 (255, 204, 188)，指甲可以用 (255, 240, 230)
    nail_color = (min(fill_color[0]+20, 255), min(fill_color[1]+20, 255), min(fill_color[2]+20, 255))
    knuckle_color = border_color # 关节用深色
    
    def get_rotated_offset(ox, oy):
        """将相对坐标 (ox, oy) 旋转并叠加到中心"""
        rx = ox * math.cos(base_rad) - oy * math.sin(base_rad)
        ry = ox * math.sin(base_rad) + oy * math.cos(base_rad)
        return cx + rx, cy + ry

    # --- 尺寸调整 ---
    palm_size = radius * 1.15
    finger_width = radius * 0.48
    finger_len = radius * 1.5
    
    # --- 定义手指配置 ---
    # (x偏移, y偏移, 相对角度, 长度系数)
    fingers = [
        (radius*0.15,  -radius*0.55, -12, 0.9),  # 食指 (略短)
        (radius*0.25,   0,           0,   1.0),  # 中指 (最长)
        (radius*0.15,   radius*0.55,  12,  0.9), # 无名指 (略短)
        (radius*0.05,   radius*1.0,   25,  0.75) # 小指 (最短, 新增) - 显得更真实
    ]
    
    # 1. 先画手指轮廓 (Border)
    for fx, fy, fang, flen in fingers:
        pos = get_rotated_offset(fx, fy)
        draw_capsule_rotated(surf, border_color, pos[0], pos[1], finger_width+4, finger_len*flen, angle_deg + fang)
    
    # 2. 画大拇指轮廓
    # 大拇指位置调整：手背向上时，大拇指根部其实在手掌侧面偏里
    thumb_pos = get_rotated_offset(-radius*0.2, -radius * 0.6)
    thumb_angle = -55 
    draw_capsule_rotated(surf, border_color, thumb_pos[0], thumb_pos[1], finger_width*1.3+4, finger_len*0.85, angle_deg + thumb_angle)

    # 3. 画掌心轮廓
    pygame.gfxdraw.aacircle(surf, int(cx), int(cy), int(palm_size), border_color)
    pygame.gfxdraw.filled_circle(surf, int(cx), int(cy), int(palm_size), border_color)

    # ==========================
    #       填充内部 (Fill)
    # ==========================
    
    # 辅助函数：画指甲
    def draw_nail(start_x, start_y, width, length, angle):
        # 指甲位置：在手指末端 80% 处
        nail_dist = length * 0.75
        nail_w = width * 0.6
        nail_h = width * 0.5 # 指甲稍微方一点
        
        rad = math.radians(angle)
        # 计算指甲中心
        nx = start_x + nail_dist * math.cos(rad)
        ny = start_y + nail_dist * math.sin(rad)
        
        # 这里简单画个小圆或胶囊当指甲
        draw_capsule_rotated(surf, nail_color, nx, ny, nail_h, nail_w * 0.2, angle)

    # 4. 填充手指 + 画指甲
    for fx, fy, fang, flen in fingers:
        pos = get_rotated_offset(fx, fy)
        current_len = finger_len * flen
        current_angle = angle_deg + fang
        
        # 填充肤色
        draw_capsule_rotated(surf, fill_color, pos[0], pos[1], finger_width, current_len, current_angle)
        
        # 画指甲 (关键！)
        draw_nail(pos[0], pos[1], finger_width, current_len, current_angle)
        
        # 画指关节 (Knuckles) - 在手指根部画一条淡淡的弧线或圆点
        # 这里简单用一个小圆点表示指关节隆起
        pygame.gfxdraw.aacircle(surf, int(pos[0]), int(pos[1]), int(finger_width*0.4), (230, 150, 130)) # 稍微深一点的肤色
        pygame.gfxdraw.filled_circle(surf, int(pos[0]), int(pos[1]), int(finger_width*0.4), (230, 150, 130))

    # 5. 填充大拇指 + 指甲
    draw_capsule_rotated(surf, fill_color, thumb_pos[0], thumb_pos[1], finger_width*1.3, finger_len*0.85, angle_deg + thumb_angle)
    draw_nail(thumb_pos[0], thumb_pos[1], finger_width*1.3, finger_len*0.85, angle_deg + thumb_angle)

    # 6. 填充掌心
    pygame.gfxdraw.aacircle(surf, int(cx), int(cy), int(palm_size-2), fill_color)
    pygame.gfxdraw.filled_circle(surf, int(cx), int(cy), int(palm_size-2), fill_color)
    
    # 7. 手背特征：掌骨线 (Metacarpal lines)
    # 在手背画两条淡淡的线，模拟肌腱，增加立体感
    for i in [-1, 1]:
        start = get_rotated_offset(-radius*0.5, i * radius*0.3)
        end = get_rotated_offset(0, i * radius*0.2)
        pygame.draw.line(surf, (230, 150, 130), start, end, 2)

# --- 1. 定义学术风格配色 (Scientific Color Palette) ---
COLORS = {
    'bg_main': (250, 250, 250),      # 极淡的灰白背景，比纯白护眼
    'grid': (230, 230, 230),         # 网格线
    'arm_fill': (255, 204, 188),     # 柔和的肤色 (Peach)
    'arm_border': (230, 74, 25),     # 手臂边缘深色，增加轮廓清晰度
    'robot': (0,0,0),          # 扁平红 (Alizarin)
    'robot_shadow': (200, 50, 50),   # 机器人阴影
    'hand_core': (46, 204, 113),     # 扁平绿 (Emerald)
    'trajectory': (52, 152, 219),    # 扁平蓝 (Peter River)
    'text': (50, 60, 80)             # 深灰字体
}

def draw_aa_circle(surf, color, center, radius):
    """画抗锯齿的实心圆"""
    x, y = int(center[0]), int(center[1])
    pygame.gfxdraw.aacircle(surf, x, y, radius, color)
    pygame.gfxdraw.filled_circle(surf, x, y, radius, color)

def draw_capsule(surf, color, start_pos, end_pos, width):
    """画胶囊形状（用于模拟手臂），比单纯的粗线好看"""
    x1, y1 = start_pos
    x2, y2 = end_pos
    length = np.hypot(x2-x1, y2-y1)
    if length == 0: return

    angle = np.arctan2(y2-y1, x2-x1)
    
    # 计算矩形的四个角
    dx = width/2 * np.sin(angle)
    dy = width/2 * np.cos(angle)
    
    # 胶囊的身体（多边形）
    points = [
        (x1 - dx, y1 + dy),
        (x2 - dx, y2 + dy),
        (x2 + dx, y2 - dy),
        (x1 + dx, y1 - dy)
    ]
    pygame.gfxdraw.aapolygon(surf, points, color)
    pygame.gfxdraw.filled_polygon(surf, points, color)
    
    # 两个端点的圆头
    draw_aa_circle(surf, color, start_pos, int(width/2))
    draw_aa_circle(surf, color, end_pos, int(width/2))

def render_aesthetic(robot_pos, hand_pos, fix_point, trajectory_points, 
                    grid_size=10, cell_size=50, window=None):
    
    # ---------------- Setup ----------------
    width_px = int(grid_size * cell_size * 1.5)
    height_px = int(grid_size * cell_size)
    
    # 如果外部没有传入 window，这里初始化（用于截图或单独测试）
    if window is None:
        if not pygame.get_init():
            pygame.init()
        window = pygame.display.set_mode((width_px, height_px))

    # 使用 Surface 进行离屏渲染（可选：如果是做视频，可以渲染2倍大小再缩小以获得超级抗锯齿）
    canvas = pygame.Surface((width_px, height_px))
    canvas.fill(COLORS['bg_main'])

    # ---------------- 1. Draw Grid (Laboratory Feel) ----------------
    # 画网格让画面更有"度量感"
    for x in range(0, width_px, cell_size):
        pygame.draw.line(canvas, COLORS['grid'], (x, 0), (x, height_px), 1)
    for y in range(0, height_px, cell_size):
        pygame.draw.line(canvas, COLORS['grid'], (0, y), (width_px, y), 1)

    # ---------------- 2. Draw Arm (Constraint) ----------------
    # 将坐标转换为像素
    fix_px = np.array(fix_point) * cell_size
    hand_px = np.array(hand_pos) * cell_size
    
    # 画手臂 (胶囊体) - 模拟人的手臂障碍
    arm_width = 40
    # 先画深色轮廓
    draw_capsule(canvas, COLORS['arm_border'], fix_px, hand_px, arm_width + 4)
    # 再画浅色内部
    draw_capsule(canvas, COLORS['arm_fill'], fix_px, hand_px, arm_width)

    # ---------------- 3. Draw Trajectory (Fading Tail) ----------------
    if len(trajectory_points) > 1:
        # 只取最近的 N 个点，避免屏幕太乱
        recent_points = trajectory_points[-50:] 
        
        for i in range(len(recent_points) - 1):
            pt1 = (int(recent_points[i][0] * cell_size), int(recent_points[i][1] * cell_size))
            pt2 = (int(recent_points[i+1][0] * cell_size), int(recent_points[i+1][1] * cell_size))
            
            # 越新的点越不透明
            alpha = int(255 * (i / len(recent_points)))
            
            # Pygame draw.line 不支持 alpha，需要画在一个带 alpha 的 surface 上
            # 或者简单点：随着 alpha 改变颜色深浅（变白）
            # 这里用一种简单的近似：绘制多重细线或实心圆
            
            # 高级画法：画圆点组成的轨迹
            draw_aa_circle(canvas, (*COLORS['trajectory'], alpha), pt1, 2)
            
            # 画线
            # 注意：这里如果想要漂亮的透明线，需要创建临时 surface，为了性能这里简化处理
            if i > len(recent_points) - 10: # 只连最后几段线
                 pygame.draw.line(canvas, COLORS['trajectory'], pt1, pt2, 2)

    # ---------------- 4. Draw Robot (The Agent) ----------------
    robot_px = np.array(robot_pos) * cell_size
    robot_radius = int(cell_size * 0.25)
    
    # 绘制阴影 (Shadow) - 增加立体感
    shadow_offset = (3, 3)
    draw_aa_circle(canvas, (200, 200, 200), robot_px + shadow_offset, robot_radius)
    
    # 绘制机器人主体
    draw_aa_circle(canvas, COLORS['robot'], robot_px, robot_radius)
    # 绘制机器人高光 (Highlight)
    draw_aa_circle(canvas, (255, 255, 255), robot_px - (robot_radius*0.3, robot_radius*0.3), int(robot_radius*0.3))

   # ---------------- 5. Draw Hand ----------------
    # 向量：从固定点 -> 手掌位置
    dx = hand_px[0] - fix_px[0]
    dy = hand_px[1] - fix_px[1]
    
    # 计算角度 (弧度 -> 度)
    # math.degrees(math.atan2(dy, dx)) 会给出向量相对于 X轴 的角度
    # 这正是我们 draw_detailed_hand 函数需要的 angle_deg
    hand_angle = math.degrees(math.atan2(dy, dx))
    
    hand_size = cell_size * 0.6 # 调整大小
    
    # 调用绘制
    draw_detailed_hand(canvas, COLORS['arm_fill'], COLORS['arm_border'], 
                      hand_px, hand_size, angle_deg=hand_angle)
    
    # 装饰：中心白点
    draw_aa_circle(canvas, (255, 255, 255), hand_px, int(cell_size * 0.08))

    # ---------------- 6. UI / Text ----------------
    font = pygame.font.SysFont("Arial", 18) # 使用系统字体，通常比默认的好看
    coord_text = f"Hand: ({hand_pos[0]:.2f}, {hand_pos[1]:.2f})"
    
    # 绘制文字背景框 (让文字看清楚)
    # text_surf = font.render(coord_text, True, COLORS['text'])
    # text_bg = pygame.Surface((text_surf.get_width()+10, text_surf.get_height()+6))
    # text_bg.fill((255, 255, 255))
    # text_bg.set_alpha(200) # 半透明背景
    
    # canvas.blit(text_bg, (10, 10))
    # canvas.blit(text_surf, (15, 13))

    # ---------------- Final Blit ----------------
    window.blit(canvas, (0, 0))
    pygame.display.flip()
    
    # 处理事件防止卡死 (保留原来的逻辑)
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            pygame.quit()
            import sys; sys.exit()
    
    return canvas # 返回 canvas 方便截图

In [1]:
import torch
import torch.nn as nn
class MetaControllerNet(nn.Module):
    def __init__(self, input_dim=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 1)
        )
    def forward(self, x): return self.net(x)
model = MetaControllerNet()
model.load_state_dict(torch.load(r"C:\Users\admin\Desktop\科研\RL\src\meta_controller.pth"))

C:\Users\admin\AppData\Local\Temp\ipykernel_34704\3659793913.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(r"C:\Users\admin\Desktop\科

<All keys matched successfully>

In [13]:
model(torch.tensor([1, 5, 0.6, 0.7]))

tensor([2.7967], grad_fn=<ViewBackward0>)

In [19]:
import pickle
with open(r"C:\Users\admin\Desktop\科研\RL\src\meta_dataset.pkl", "rb") as f:
    trajectory_points = pickle.load(f)
index = 70
trajectory_points[0][index], trajectory_points[1][index]

(array([0.45774237, 3.40980243, 0.91786527, 0.83359858]),
 np.float64(2.8000000000000016))